# MC dropout uncertainty and attribution
### ADNI 36-month MCI prognosis — fixed 3MT–TMC fusion

This analysis adds **Monte Carlo (MC) dropout** as a complementary uncertainty quantification method and compares it directly with the existing evidential and conflict-aware uncertainty analysis.

The default experiment is the fixed-equal-fusion model because the dissertation's detailed uncertainty attribution was performed on that same model. This avoids changing the classifier while changing the uncertainty method.


## Why MC dropout is appropriate here

The trained architecture already contains ordinary neural dropout in the tabular encoders, MRI transformer, cascaded transformer, and evidential heads. MC dropout therefore requires **repeated inference, not retraining**, for this first comparison.

the architecture also has 50% *modality dropout* during training. Calling `model.train()` at test time would reactivate that mechanism and randomly delete clinical branches. The helper below keeps the top-level model in evaluation mode and activates only standard dropout/attention submodules.


In [ ]:

# 1. Imports, Drive mount, and experiment configuration

from pathlib import Path
from itertools import combinations
import copy
import json
import math
import random
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from scipy.stats import spearmanr

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped (not running in Colab).")

# Project paths
PROJECT_ROOT = Path("/content/drive/MyDrive/adni_mri")
MODEL_ROOT = PROJECT_ROOT / "models" / "3mt_tmc_evidential"
TASK_NAME = "mci_prognosis"

# This is deliberately the SAME fixed-equal-fusion experiment used for the
# detailed uncertainty/conflict attribution in the dissertation. That makes
# the comparison with MC dropout apples-to-apples.
EXPERIMENT_NAME = "gated_cmt_fixed_equal_fusion_md050"
EXPERIMENT_ROOT = MODEL_ROOT / "experiments" / EXPERIMENT_NAME / TASK_NAME

INPUT_ROOT = MODEL_ROOT / "final_task_ready_inputs" / TASK_NAME
SCHEMA_PATH = MODEL_ROOT / "manifests" / "final_model_input_schema.json"

OUTPUT_ROOT = (
    MODEL_ROOT
    / "experiment_comparisons"
    / "mc_dropout_vs_evidential_fixed_md050"
)
TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = OUTPUT_ROOT / "figures"
for d in [OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Runtime controls
# Start with PILOT_MODE=True. After the first fold runs successfully, set it
# to False for the five-fold comparison.
PILOT_MODE = True

OUTER_FOLDS = (0,) if PILOT_MODE else (0, 1, 2, 3, 4)
MC_SAMPLES = 10 if PILOT_MODE else 30
BATCH_SIZE = 2
NUM_WORKERS = 0
CLASSIFICATION_THRESHOLD = 0.50
NUMBER_OF_CLASSES = 2

# Optional leave-one-branch-out attribution. This is substantially more
# expensive because every available branch requires repeated stochastic
# inference. Run it after the main MC-dropout analysis works.
RUN_LOBO_ATTRIBUTION = False
LOBO_MC_SAMPLES = 10

# Bootstrap for paired uncertainty-score comparisons.
BOOTSTRAP_ITERATIONS = 2000
BOOTSTRAP_SEED = 20260928

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Pilot mode:", PILOT_MODE)
print("Folds:", OUTER_FOLDS)
print("MC samples:", MC_SAMPLES)

# Reference values from the dissertation's fixed-fusion attribution analysis.
# The full five-fold deterministic reconstruction should be close to these.
REFERENCE_EVIDENTIAL_ERROR_AUC = 0.6572
REFERENCE_CONFLICT_ERROR_AUC = 0.7214


In [ ]:

# 2. Shared constants and dataset contract

MRI_SPATIAL_SHAPE = (177, 213, 183)
MRI_TENSOR_SHAPE = (1, *MRI_SPATIAL_SHAPE)

MODALITY_EMBEDDING_DIM = 64

MODALITY_ORDER = [
    "demographics",
    "cognitive_functional",
    "csf",
    "plasma",
    "apoe",
    "mri",
]

THREE_MT_CASCADE_ORDER = [
    "demographics",
    "apoe",
    "cognitive_functional",
    "csf",
    "plasma",
    "mri",
]

PROGNOSIS_CLASS_ORDER = ["sMCI", "pMCI"]

dataset_column_contract = {
    "demographics": {
        "continuous": [
            "DEMOGRAPHICS__AGE_AT_BASELINE__Z",
            "DEMOGRAPHICS__PTEDUCAT_CLEAN__Z",
        ],
        "categorical": [
            "DEMOGRAPHICS__PTGENDER_CLEAN__IDX",
            "DEMOGRAPHICS__PTHAND_CLEAN__IDX",
        ],
        "feature_masks": [
            "FEATURE_MASK__DEMOGRAPHICS_AGE_AT_BASELINE",
            "FEATURE_MASK__DEMOGRAPHICS_PTEDUCAT_CLEAN",
            "FEATURE_MASK__DEMOGRAPHICS_PTGENDER_CLEAN",
            "FEATURE_MASK__DEMOGRAPHICS_PTHAND_CLEAN",
        ],
        "branch_mask": "BRANCH_MASK__DEMOGRAPHICS",
    },

    "cognitive_functional": {
        "continuous": [
            "ADAS__TOTSCORE__Z",
            "ADAS__TOTAL13__Z",
            "MMSE__MMSE_TOTAL_SCORE__Z",
            "MMSE__MMSE_ORIENTATION_SCORE__Z",
            "MMSE__MMSE_REGISTRATION_SCORE__Z",
            "MMSE__MMSE_ATTENTION_SCORE__Z",
            "MMSE__MMSE_DELAYED_RECALL_SCORE__Z",
            "MMSE__MMSE_LANGUAGE_COMMAND_SCORE__Z",
            "FAQ__FAQTOTAL__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__ADAS_TOTSCORE",
            "FEATURE_MASK__ADAS_TOTAL13",
            "FEATURE_MASK__MMSE_MMSE_TOTAL_SCORE",
            "FEATURE_MASK__MMSE_MMSE_ORIENTATION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_REGISTRATION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_ATTENTION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_DELAYED_RECALL_SCORE",
            "FEATURE_MASK__MMSE_MMSE_LANGUAGE_COMMAND_SCORE",
            "FEATURE_MASK__FAQ_FAQTOTAL",
        ],
        "branch_mask": "BRANCH_MASK__COGNITIVE_FUNCTIONAL",
    },

    "csf": {
        "continuous": [
            "CSF__ABETA40__Z",
            "CSF__ABETA42__Z",
            "CSF__TAU__Z",
            "CSF__PTAU__Z",
            "CSF__ABETA42_40_RATIO__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__CSF_ABETA40",
            "FEATURE_MASK__CSF_ABETA42",
            "FEATURE_MASK__CSF_TAU",
            "FEATURE_MASK__CSF_PTAU",
            "FEATURE_MASK__CSF_ABETA42_40_RATIO",
        ],
        "branch_mask": "BRANCH_MASK__CSF",
    },

    "plasma": {
        "continuous": [
            "PLASMA__pT217_F__Z",
            "PLASMA__AB42_F__Z",
            "PLASMA__AB40_F__Z",
            "PLASMA__AB42_AB40_F__Z",
            "PLASMA__pT217_AB42_F__Z",
            "PLASMA__NfL_Q__Z",
            "PLASMA__GFAP_Q__Z",
            "PLASMA__NfL_F__Z",
            "PLASMA__GFAP_F__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__PLASMA_pT217_F",
            "FEATURE_MASK__PLASMA_AB42_F",
            "FEATURE_MASK__PLASMA_AB40_F",
            "FEATURE_MASK__PLASMA_AB42_AB40_F",
            "FEATURE_MASK__PLASMA_pT217_AB42_F",
            "FEATURE_MASK__PLASMA_NfL_Q",
            "FEATURE_MASK__PLASMA_GFAP_Q",
            "FEATURE_MASK__PLASMA_NfL_F",
            "FEATURE_MASK__PLASMA_GFAP_F",
        ],
        "branch_mask": "BRANCH_MASK__PLASMA",
    },

    "apoe": {
        "continuous": [],
        "categorical": [
            "APOE__APOE4_ALLELE_COUNT__IDX",
        ],
        "feature_masks": [
            "FEATURE_MASK__APOE_APOE4_ALLELE_COUNT",
        ],
        "branch_mask": "BRANCH_MASK__APOE",
    },

    "mri": {
        "path": "MRI__NORMALIZED_T1_NPY_PATH",
        "branch_mask": "BRANCH_MASK__MRI",
    },
}


In [ ]:

# 3. Architecture definitions copied from the current fixed-fusion notebook
#
# The model definition below is intentionally kept structurally identical to
# the trained fixed-equal-fusion model so the saved state_dict checkpoints can
# be loaded directly.
#
# IMPORTANT FOR MC DROPOUT:
# The model also contains 50% *modality dropout* during training. We must NOT
# reactivate the whole model with model.train() at test time, because that
# would randomly delete clinical branches and would confound MC-dropout
# uncertainty with missing-modality perturbation.
#

class ADNIMultimodalDataset(Dataset):
    """
    PyTorch dataset for the prepared ADNI multimodal tables.

    Each participant is returned as a dictionary containing:
    - identifiers;
    - target;
    - separate modality inputs;
    - branch-level masks;
    - feature-level masks.

    MRI arrays are loaded lazily from the prepared NumPy paths.
    """

    def __init__(
        self,
        dataframe,
        column_contract,
        branch_mask_order,
        feature_mask_order,
        target_column,
        load_mri=True,
    ):
        # Reset the row index so that PyTorch sample indices map
        # directly to positional rows in this dataset.
        self.dataframe = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        # Retain the prepared column organisation rather than
        # deriving new feature groups from column-name patterns.
        self.column_contract = column_contract

        # Preserve the mask order from the final
        # model-input schema.
        self.branch_mask_order = list(branch_mask_order)
        self.feature_mask_order = list(feature_mask_order)

        self.target_column = target_column

        # This option allows scalar-only experiments and dataset
        # inspection without reading the large MRI arrays.
        self.load_mri = load_mri

    def __len__(self):
        return len(self.dataframe)

    @staticmethod
    def _continuous_tensor(row, columns):
        """
        Convert prepared continuous values to a float32 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.float32)
        )

        return torch.from_numpy(values)

    @staticmethod
    def _categorical_tensor(row, columns):
        """
        Convert prepared categorical indices to an int64 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.int64)
        )

        return torch.from_numpy(values)

    @staticmethod
    def _mask_tensor(row, columns):
        """
        Convert prepared binary masks to a float32 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.float32)
        )

        return torch.from_numpy(values)

    def _load_mri_tensor(
        self,
        mri_path,
        mri_branch_mask,
    ):
        """
        Load one prepared MRI array or return a masked placeholder.
        """

        # A participant without MRI remains in the dataset.
        if float(mri_branch_mask) == 0.0:
            return torch.zeros(
                MRI_TENSOR_SHAPE,
                dtype=torch.float32,
            )

        # Scalar-only inspection can skip disk loading while
        # preserving the same output structure.
        if not self.load_mri:
            return torch.zeros(
                MRI_TENSOR_SHAPE,
                dtype=torch.float32,
            )

        # An available MRI branch should have a prepared path.
        if pd.isna(mri_path):
            raise ValueError(
                "MRI branch mask is 1, but the MRI path is missing."
            )

        mri_path = Path(str(mri_path))

        if not mri_path.exists():
            raise FileNotFoundError(
                f"Prepared MRI array was not found: {mri_path}"
            )

        # Load the already normalised NumPy volume without
        # applying any additional preprocessing.
        mri_array = np.load(
            mri_path,
            allow_pickle=False,
        )

        if mri_array.shape != MRI_SPATIAL_SHAPE:
            raise ValueError(
                "Unexpected MRI shape for "
                f"{mri_path}: {mri_array.shape}"
            )

        # Ensure float32 representation and add the channel axis:
        # (177, 213, 183) -> (1, 177, 213, 183).
        mri_array = np.asarray(
            mri_array,
            dtype=np.float32,
        )

        mri_array = np.expand_dims(
            mri_array,
            axis=0,
        )

        return torch.from_numpy(mri_array)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        modalities = {}

        # Demographics

        demographics_contract = self.column_contract[
            "demographics"
        ]

        modalities["demographics"] = {
            "continuous": self._continuous_tensor(
                row,
                demographics_contract["continuous"],
            ),

            "categorical": self._categorical_tensor(
                row,
                demographics_contract["categorical"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                demographics_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    demographics_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }

        # Cognitive and functional measures

        cognitive_contract = self.column_contract[
            "cognitive_functional"
        ]

        modalities["cognitive_functional"] = {
            "continuous": self._continuous_tensor(
                row,
                cognitive_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                cognitive_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    cognitive_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }

        # CSF

        csf_contract = self.column_contract["csf"]

        modalities["csf"] = {
            "continuous": self._continuous_tensor(
                row,
                csf_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                csf_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    csf_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }

        # Plasma

        plasma_contract = self.column_contract["plasma"]

        modalities["plasma"] = {
            "continuous": self._continuous_tensor(
                row,
                plasma_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                plasma_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    plasma_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }

        # APOE

        apoe_contract = self.column_contract["apoe"]

        modalities["apoe"] = {
            "categorical": self._categorical_tensor(
                row,
                apoe_contract["categorical"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                apoe_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    apoe_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }

        # MRI

        mri_contract = self.column_contract["mri"]

        mri_branch_mask = row[
            mri_contract["branch_mask"]
        ]

        mri_path = row[
            mri_contract["path"]
        ]

        modalities["mri"] = {
            "image": self._load_mri_tensor(
                mri_path=mri_path,
                mri_branch_mask=mri_branch_mask,
            ),

            "branch_mask": torch.tensor(
                mri_branch_mask,
                dtype=torch.float32,
            ),
        }

        # Complete sample

        sample = {
            "rid": int(row["RID"]),
            "ptid": str(row["PTID"]),

            "target": torch.tensor(
                int(row[self.target_column]),
                dtype=torch.long,
            ),

            "modalities": modalities,

            "branch_masks": self._mask_tensor(
                row,
                self.branch_mask_order,
            ),

            "feature_masks": self._mask_tensor(
                row,
                self.feature_mask_order,
            ),
        }

        return sample

class MaskAwareScalarEncoder(nn.Module):
    """
    Encode continuous scalar features together with their
    prepared feature-observation masks.
    """

    def __init__(
        self,
        value_dim,
        mask_dim,
        output_dim,
        hidden_dim=64,
        dropout=0.20,
    ):
        super().__init__()

        input_dim = value_dim + mask_dim

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )

    def forward(
        self,
        values,
        feature_mask,
    ):
        # Pass both the prepared values and their masks so that
        # missing placeholders are not treated as genuine observations.
        inputs = torch.cat(
            [
                values,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)

class DemographicsEncoder(nn.Module):
    """
    Encode continuous and categorical demographic predictors.
    """

    def __init__(
        self,
        output_dim,
        dropout=0.20,
    ):
        super().__init__()

        # Sex:
        # 0 = missing or unseen;
        # 1 and 2 = observed prepared categories.
        self.sex_embedding = nn.Embedding(
            num_embeddings=3,
            embedding_dim=4,
            padding_idx=0,
        )

        # Handedness:
        # 0 = missing or unseen;
        # 1 and 2 = observed prepared categories.
        self.handedness_embedding = nn.Embedding(
            num_embeddings=3,
            embedding_dim=4,
            padding_idx=0,
        )

        # Input components:
        # 2 continuous values;
        # 4-dimensional sex embedding;
        # 4-dimensional handedness embedding;
        # 4 feature masks.
        input_dim = 2 + 4 + 4 + 4

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                48,
            ),
            nn.LayerNorm(48),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                48,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )

    def forward(
        self,
        continuous,
        categorical,
        feature_mask,
    ):
        sex_index = categorical[:, 0]
        handedness_index = categorical[:, 1]

        sex_representation = self.sex_embedding(
            sex_index
        )

        handedness_representation = (
            self.handedness_embedding(
                handedness_index
            )
        )

        inputs = torch.cat(
            [
                continuous,
                sex_representation,
                handedness_representation,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)

class APOEEncoder(nn.Module):
    """
    Encode the prepared APOE epsilon-4 allele-count index.
    """

    def __init__(
        self,
        output_dim,
        dropout=0.10,
    ):
        super().__init__()

        # Prepared APOE indices:
        # 0 = missing;
        # 1 = zero epsilon-4 alleles;
        # 2 = one epsilon-4 allele;
        # 3 = two epsilon-4 alleles.
        self.apoe_embedding = nn.Embedding(
            num_embeddings=4,
            embedding_dim=8,
            padding_idx=0,
        )

        self.network = nn.Sequential(
            nn.Linear(
                8 + 1,
                32,
            ),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                32,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )

    def forward(
        self,
        categorical,
        feature_mask,
    ):
        apoe_index = categorical[:, 0]

        apoe_representation = self.apoe_embedding(
            apoe_index
        )

        inputs = torch.cat(
            [
                apoe_representation,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)

class ResidualDownsampleBlock3D(nn.Module):
    """
    Downsample a three-dimensional feature map and learn a
    residual representation at the new channel width.
    """

    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        # The original 3MT image encoder applies spatial
        # downsampling before the residual convolutional paths.
        self.pool = nn.MaxPool3d(
            kernel_size=2,
            stride=2,
        )

        self.main_path = nn.Sequential(
            nn.Conv3d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
            nn.GELU(),

            nn.Conv3d(
                in_channels=out_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
        )

        # A point-wise convolution aligns the residual path with
        # the new number of channels.
        self.residual_path = nn.Conv3d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=1,
            stride=1,
            bias=False,
        )

        self.activation = nn.GELU()

    def forward(self, inputs):
        pooled_inputs = self.pool(
            inputs
        )

        main_features = self.main_path(
            pooled_inputs
        )

        residual_features = self.residual_path(
            pooled_inputs
        )

        return self.activation(
            main_features
            + residual_features
        )

class MRIEncoder3D(nn.Module):
    """
    Encode the prepared full-volume MRI using a 3D CNN followed
    by a patch-wise transformer encoder.
    """

    def __init__(
        self,
        output_dim,
        input_shape=MRI_SPATIAL_SHAPE,
        patch_embedding_dim=256,
        transformer_heads=8,
        transformer_layers=1,
        transformer_feedforward_dim=512,
        dropout=0.20,
    ):
        super().__init__()

        if patch_embedding_dim % transformer_heads != 0:
            raise ValueError(
                "The MRI patch-embedding dimension must be "
                "divisible by the number of attention heads."
            )

        self.input_shape = tuple(
            input_shape
        )

        self.patch_embedding_dim = (
            patch_embedding_dim
        )

        # Initial convolutional stem

        # Use two initial 3D convolutions, following the broad
        # structure shown in the 3MT image encoder.
        #
        # The first convolution uses stride two because the prepared
        # MRI volumes are larger than the original 3MT inputs.
        self.stem = nn.Sequential(
            nn.Conv3d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),

            nn.Conv3d(
                in_channels=16,
                out_channels=16,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),
        )

        # Four residual downsampling blocks

        self.residual_blocks = nn.Sequential(
            ResidualDownsampleBlock3D(
                in_channels=16,
                out_channels=32,
            ),

            ResidualDownsampleBlock3D(
                in_channels=32,
                out_channels=64,
            ),

            ResidualDownsampleBlock3D(
                in_channels=64,
                out_channels=128,
            ),

            ResidualDownsampleBlock3D(
                in_channels=128,
                out_channels=256,
            ),
        )

        # Determine the resulting patch grid

        # The stride-two stem convolution applies ceiling division
        # by two for these kernel and padding settings.
        stem_shape = tuple(
            math.ceil(dimension / 2)
            for dimension in self.input_shape
        )

        # Each of the four MaxPool3d layers applies floor division
        # by two.
        patch_grid_shape = stem_shape

        for _ in range(4):
            patch_grid_shape = tuple(
                dimension // 2
                for dimension in patch_grid_shape
            )

        if any(
            dimension < 1
            for dimension in patch_grid_shape
        ):
            raise ValueError(
                "The MRI input becomes too small after "
                "convolutional downsampling."
            )

        self.patch_grid_shape = (
            patch_grid_shape
        )

        self.number_of_patches = math.prod(
            patch_grid_shape
        )

        # Learned positional embeddings

        # Each location in the final 3D feature map becomes one
        # transformer patch token.
        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                self.number_of_patches,
                patch_embedding_dim,
            )
        )

        nn.init.trunc_normal_(
            self.position_embedding,
            std=0.02,
        )

        # Patch-wise transformer encoder

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=patch_embedding_dim,
            nhead=transformer_heads,
            dim_feedforward=transformer_feedforward_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer=transformer_layer,
            num_layers=transformer_layers,
            norm=nn.LayerNorm(
                patch_embedding_dim
            ),
        )

        # Projection to the shared modality dimension

        self.projection = nn.Sequential(
            nn.Linear(
                patch_embedding_dim,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, image):
        # Expected image shape:
        # (batch_size, 1, 177, 213, 183)
        feature_map = self.stem(
            image
        )

        feature_map = self.residual_blocks(
            feature_map
        )

        # Expected feature-map organisation:
        # (batch_size, 256, depth, height, width)
        batch_size, channels, depth, height, width = (
            feature_map.shape
        )

        actual_patch_count = (
            depth
            * height
            * width
        )

        if actual_patch_count != self.number_of_patches:
            raise ValueError(
                "Unexpected MRI patch count. "
                f"Expected {self.number_of_patches}, "
                f"but obtained {actual_patch_count}."
            )

        # Flatten the spatial locations into patch tokens:
        #
        # (B, C, D, H, W)
        # -> (B, C, N)
        # -> (B, N, C)
        patch_tokens = (
            feature_map
            .flatten(start_dim=2)
            .transpose(1, 2)
        )

        # Add learned positional information before modelling
        # relationships between the 3D patch representations.
        patch_tokens = (
            patch_tokens
            + self.position_embedding
        )

        transformed_tokens = (
            self.transformer_encoder(
                patch_tokens
            )
        )

        # The paper applies patch-wise average pooling before the
        # final linear projection.
        pooled_representation = (
            transformed_tokens.mean(
                dim=1
            )
        )

        return self.projection(
            pooled_representation
        )

class ADNIModalityEncoders(nn.Module):
    """
    Produce one common-dimensional representation per modality.
    """

    def __init__(
        self,
        output_dim=MODALITY_EMBEDDING_DIM,
    ):
        super().__init__()

        self.demographics = DemographicsEncoder(
            output_dim=output_dim,
        )

        self.cognitive_functional = (
            MaskAwareScalarEncoder(
                value_dim=9,
                mask_dim=9,
                hidden_dim=96,
                output_dim=output_dim,
            )
        )

        self.csf = MaskAwareScalarEncoder(
            value_dim=5,
            mask_dim=5,
            hidden_dim=64,
            output_dim=output_dim,
        )

        self.plasma = MaskAwareScalarEncoder(
            value_dim=9,
            mask_dim=9,
            hidden_dim=96,
            output_dim=output_dim,
        )

        self.apoe = APOEEncoder(
            output_dim=output_dim,
        )

        self.mri = MRIEncoder3D(
            output_dim=output_dim,
            input_shape=MRI_SPATIAL_SHAPE,
            patch_embedding_dim=256,
            transformer_heads=8,
            transformer_layers=1,
            transformer_feedforward_dim=512,
            dropout=0.20,
        )

    def forward(self, modalities):
        representations = {}

        representations["demographics"] = (
            self.demographics(
                continuous=modalities[
                    "demographics"
                ]["continuous"],

                categorical=modalities[
                    "demographics"
                ]["categorical"],

                feature_mask=modalities[
                    "demographics"
                ]["feature_mask"],
            )
        )

        representations["cognitive_functional"] = (
            self.cognitive_functional(
                values=modalities[
                    "cognitive_functional"
                ]["continuous"],

                feature_mask=modalities[
                    "cognitive_functional"
                ]["feature_mask"],
            )
        )

        representations["csf"] = self.csf(
            values=modalities[
                "csf"
            ]["continuous"],

            feature_mask=modalities[
                "csf"
            ]["feature_mask"],
        )

        representations["plasma"] = self.plasma(
            values=modalities[
                "plasma"
            ]["continuous"],

            feature_mask=modalities[
                "plasma"
            ]["feature_mask"],
        )

        representations["apoe"] = self.apoe(
            categorical=modalities[
                "apoe"
            ]["categorical"],

            feature_mask=modalities[
                "apoe"
            ]["feature_mask"],
        )

        representations["mri"] = self.mri(
            modalities[
                "mri"
            ]["image"]
        )

        return representations

class MaskedADNIModalityEncoders(nn.Module):
    """
    Run the six modality encoders and suppress representations
    from unavailable branches.
    """

    def __init__(
        self,
        output_dim=MODALITY_EMBEDDING_DIM,
    ):
        super().__init__()

        self.output_dim = output_dim

        self.encoders = ADNIModalityEncoders(
            output_dim=output_dim,
        )

    @staticmethod
    def _apply_branch_mask(
        representation,
        branch_mask,
    ):
        """
        Multiply each participant's representation by the
        corresponding scalar branch-availability mask.
        """

        # representation:
        #     (batch_size, embedding_dim)
        #
        # branch_mask:
        #     (batch_size,)
        #
        # Add a final dimension so broadcasting is explicit:
        #     (batch_size,) -> (batch_size, 1)
        expanded_mask = branch_mask.unsqueeze(-1)

        return representation * expanded_mask

    def forward(self, modalities):
        # Get the ordinary encoder outputs.
        raw_representations = self.encoders(
            modalities
        )

        masked_representations = {}

        # Mask every unavailable branch using its own
        # prepared branch-level mask.
        for modality_name in MODALITY_ORDER:

            branch_mask = modalities[
                modality_name
            ]["branch_mask"]

            masked_representations[modality_name] = (
                self._apply_branch_mask(
                    representation=raw_representations[
                        modality_name
                    ],
                    branch_mask=branch_mask,
                )
            )

        # Return both versions for later interpretation and
        # debugging. Only the masked representations should enter
        # multimodal interaction and fusion.
        return {
            "raw": raw_representations,
            "masked": masked_representations,
        }

class CascadedModalityTransformer(nn.Module):
    """
    Apply query self-attention and inject one modality through
    cross-attention.
    """

    def __init__(
        self,
        embedding_dim,
        number_of_heads=4,
        dropout=0.10,
    ):
        super().__init__()

        self.self_attention_norm = nn.LayerNorm(
            embedding_dim
        )

        self.self_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=number_of_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.self_attention_dropout = nn.Dropout(
            dropout
        )

        self.cross_attention_norm = nn.LayerNorm(
            embedding_dim
        )

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=number_of_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.cross_attention_dropout = nn.Dropout(
            dropout
        )

        self.output_norm = nn.LayerNorm(
            embedding_dim
        )

    def forward(
        self,
        latent_query,
        modality_embedding,
    ):
        normalised_query = self.self_attention_norm(
            latent_query
        )

        self_attention_output, self_attention_weights = (
            self.self_attention(
                query=normalised_query,
                key=normalised_query,
                value=normalised_query,
                need_weights=True,
                average_attn_weights=False,
            )
        )

        self_attended_query = (
            latent_query
            + self.self_attention_dropout(
                self_attention_output
            )
        )

        normalised_self_query = self.cross_attention_norm(
            self_attended_query
        )

        cross_attention_output, cross_attention_weights = (
            self.cross_attention(
                query=normalised_self_query,
                key=modality_embedding,
                value=modality_embedding,
                need_weights=True,
                average_attn_weights=False,
            )
        )

        updated_query = (
            self_attended_query
            + self.cross_attention_dropout(
                cross_attention_output
            )
        )

        updated_query = self.output_norm(
            updated_query
        )

        return {
            "updated_query": updated_query,
            "self_attention_weights": self_attention_weights,
            "cross_attention_weights": cross_attention_weights,
        }

class ThreeMTCascade(nn.Module):
    """
    Refine one learned latent query through the six CMT stages.

    A stage uses its candidate update only when the corresponding
    effective branch mask is one. Otherwise, the previous query is
    preserved exactly.
    """

    def __init__(
        self,
        embedding_dim=MODALITY_EMBEDDING_DIM,
        modality_order=MODALITY_ORDER,
        cascade_order=THREE_MT_CASCADE_ORDER,
        number_of_heads=4,
        dropout=0.10,
    ):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.modality_order = list(modality_order)
        self.cascade_order = list(cascade_order)

        self.modality_to_mask_index = {
            modality_name: modality_index
            for modality_index, modality_name in enumerate(
                self.modality_order
            )
        }

        self.learned_latent_query = nn.Parameter(
            torch.empty(
                1,
                1,
                embedding_dim,
            )
        )

        nn.init.normal_(
            self.learned_latent_query,
            mean=0.0,
            std=0.02,
        )

        self.cmt_blocks = nn.ModuleDict(
            {
                modality_name:
                    CascadedModalityTransformer(
                        embedding_dim=embedding_dim,
                        number_of_heads=number_of_heads,
                        dropout=dropout,
                    )

                for modality_name in self.cascade_order
            }
        )

    def forward(
        self,
        masked_representations,
        branch_masks,
    ):
        first_modality = self.cascade_order[0]

        batch_size = masked_representations[
            first_modality
        ].shape[0]

        latent_query = self.learned_latent_query.expand(
            batch_size,
            -1,
            -1,
        )

        stage_queries = {}
        self_attention_weights = {}
        cross_attention_weights = {}

        for modality_name in self.cascade_order:
            previous_query = latent_query

            modality_token = masked_representations[
                modality_name
            ].unsqueeze(1)

            stage_output = self.cmt_blocks[
                modality_name
            ](
                latent_query=previous_query,
                modality_embedding=modality_token,
            )

            candidate_query = stage_output[
                "updated_query"
            ]

            modality_index = self.modality_to_mask_index[
                modality_name
            ]

            availability = branch_masks[
                :,
                modality_index,
            ].view(
                -1,
                1,
                1,
            ).to(
                dtype=previous_query.dtype
            )

            latent_query = (
                availability * candidate_query
                + (1.0 - availability) * previous_query
            )

            stage_queries[modality_name] = latent_query

            self_attention_weights[modality_name] = (
                stage_output[
                    "self_attention_weights"
                ]
            )

            cross_attention_weights[modality_name] = (
                stage_output[
                    "cross_attention_weights"
                ]
            )

        joint_representation = latent_query.squeeze(
            dim=1
        )

        return {
            "joint_representation":
                joint_representation,

            "stage_queries":
                stage_queries,

            "self_attention_weights":
                self_attention_weights,

            "cross_attention_weights":
                cross_attention_weights,
        }

class EvidentialClassificationHead(nn.Module):
    """
    Convert one modality representation into non-negative class
    evidence and the corresponding Dirichlet opinion.
    """

    def __init__(
        self,
        input_dim,
        number_of_classes,
        hidden_dim=32,
        dropout=0.10,
    ):
        super().__init__()

        self.number_of_classes = number_of_classes

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                number_of_classes,
            ),
        )

    def forward(
        self,
        representation,
        branch_mask,
    ):
        # Use Softplus to obtain non-negative evidence while
        # retaining smooth gradients.
        raw_evidence = F.softplus(
            self.network(
                representation
            )
        )

        # An unavailable modality must not contribute evidence.
        effective_evidence = (
            raw_evidence
            * branch_mask.unsqueeze(-1)
        )

        # Evidence plus one defines the Dirichlet parameters.
        alpha = effective_evidence + 1.0

        strength = alpha.sum(
            dim=-1,
            keepdim=True,
        )

        probabilities = (
            alpha
            / strength
        )

        uncertainty = (
            self.number_of_classes
            / strength
        )

        return {
            "raw_evidence": raw_evidence,
            "evidence": effective_evidence,
            "alpha": alpha,
            "strength": strength,
            "belief": effective_evidence / strength,
            "probabilities": probabilities,
            "uncertainty": uncertainty,
        }

class IndependentModalityEvidentialHeads(nn.Module):
    """
    Produce one independent Dirichlet opinion per modality before
    any 3MT cross-modal interaction.
    """

    def __init__(
        self,
        modality_order,
        input_dim,
        number_of_classes,
    ):
        super().__init__()

        self.modality_order = list(
            modality_order
        )

        self.number_of_classes = (
            number_of_classes
        )

        self.heads = nn.ModuleDict(
            {
                modality_name:
                    EvidentialClassificationHead(
                        input_dim=input_dim,
                        number_of_classes=number_of_classes,
                        hidden_dim=32,
                        dropout=0.10,
                    )

                for modality_name in self.modality_order
            }
        )

    def forward(
        self,
        modality_representations,
        modalities,
    ):
        opinions = {}

        for modality_name in self.modality_order:

            branch_mask = modalities[
                modality_name
            ]["branch_mask"]

            opinions[modality_name] = self.heads[
                modality_name
            ](
                representation=modality_representations[
                    modality_name
                ],
                branch_mask=branch_mask,
            )

        return opinions

class TMCFusion(nn.Module):
    """
    Fuse independent Dirichlet modality opinions using the
    reduced Dempster-Shafer combination rule used by TMC.
    """

    def __init__(
        self,
        number_of_classes,
        modality_order,
        numerical_epsilon=1e-8,
    ):
        super().__init__()

        self.number_of_classes = (
            number_of_classes
        )

        self.modality_order = list(
            modality_order
        )

        self.numerical_epsilon = (
            numerical_epsilon
        )

    def _dirichlet_to_opinion(
        self,
        alpha,
    ):
        """
        Convert Dirichlet parameters into belief masses and
        one uncertainty mass.
        """

        strength = alpha.sum(
            dim=-1,
            keepdim=True,
        )

        evidence = alpha - 1.0

        belief = evidence / strength

        uncertainty = (
            self.number_of_classes
            / strength
        )

        return {
            "evidence": evidence,
            "strength": strength,
            "belief": belief,
            "uncertainty": uncertainty,
        }

    def _combine_two(
        self,
        alpha_a,
        alpha_b,
    ):
        """
        Combine two batches of Dirichlet opinions.

        Both inputs have shape:
        (batch_size, number_of_classes).
        """

        opinion_a = self._dirichlet_to_opinion(
            alpha_a
        )

        opinion_b = self._dirichlet_to_opinion(
            alpha_b
        )

        belief_a = opinion_a["belief"]
        belief_b = opinion_b["belief"]

        uncertainty_a = opinion_a[
            "uncertainty"
        ]

        uncertainty_b = opinion_b[
            "uncertainty"
        ]

        # Conflict mass

        # The outer product contains every pairwise combination
        # between class beliefs from the two opinions.
        belief_outer_product = (
            belief_a.unsqueeze(-1)
            * belief_b.unsqueeze(-2)
        )

        total_belief_product = (
            belief_outer_product.sum(
                dim=(-2, -1)
            )
        )

        same_class_agreement = (
            torch.diagonal(
                belief_outer_product,
                dim1=-2,
                dim2=-1,
            )
            .sum(dim=-1)
        )

        # Conflict contains products assigned to different classes.
        conflict = (
            total_belief_product
            - same_class_agreement
        )

        normalisation = (
            1.0
            - conflict
        ).clamp_min(
            self.numerical_epsilon
        ).unsqueeze(-1)

        # Fused belief and uncertainty masses

        fused_belief = (
            belief_a * belief_b
            + belief_a * uncertainty_b
            + belief_b * uncertainty_a
        ) / normalisation

        fused_uncertainty = (
            uncertainty_a
            * uncertainty_b
        ) / normalisation

        # Recover the fused Dirichlet opinion

        fused_strength = (
            self.number_of_classes
            / fused_uncertainty.clamp_min(
                self.numerical_epsilon
            )
        )

        fused_evidence = (
            fused_belief
            * fused_strength
        )

        fused_alpha = (
            fused_evidence
            + 1.0
        )

        fused_probabilities = (
            fused_alpha
            / fused_alpha.sum(
                dim=-1,
                keepdim=True,
            )
        )

        return {
            "alpha": fused_alpha,
            "evidence": fused_evidence,
            "belief": fused_belief,
            "uncertainty": fused_uncertainty,
            "strength": fused_strength,
            "probabilities": fused_probabilities,
            "conflict": conflict.unsqueeze(-1),
        }

    def forward(
        self,
        modality_opinions,
    ):
        """
        Sequentially combine the modality-specific opinions in
        the fixed modality order.
        """

        first_modality = self.modality_order[0]

        fused_alpha = modality_opinions[
            first_modality
        ]["alpha"]

        fusion_history = {}

        # Retain the starting opinion so that the complete fusion
        # sequence can later be inspected.
        first_opinion = self._dirichlet_to_opinion(
            fused_alpha
        )

        fusion_history[first_modality] = {
            "alpha": fused_alpha,
            "belief": first_opinion["belief"],
            "uncertainty": first_opinion[
                "uncertainty"
            ],
            "conflict": torch.zeros(
                fused_alpha.shape[0],
                1,
                dtype=fused_alpha.dtype,
                device=fused_alpha.device,
            ),
        }

        for modality_name in self.modality_order[1:]:

            next_alpha = modality_opinions[
                modality_name
            ]["alpha"]

            combined = self._combine_two(
                alpha_a=fused_alpha,
                alpha_b=next_alpha,
            )

            fused_alpha = combined["alpha"]

            fusion_history[modality_name] = {
                "alpha": combined["alpha"],
                "belief": combined["belief"],
                "uncertainty": combined[
                    "uncertainty"
                ],
                "conflict": combined["conflict"],
            }

        # Final fused opinion

        final_strength = fused_alpha.sum(
            dim=-1,
            keepdim=True,
        )

        final_evidence = (
            fused_alpha
            - 1.0
        )

        final_belief = (
            final_evidence
            / final_strength
        )

        final_uncertainty = (
            self.number_of_classes
            / final_strength
        )

        final_probabilities = (
            fused_alpha
            / final_strength
        )

        return {
            "alpha": fused_alpha,
            "evidence": final_evidence,
            "belief": final_belief,
            "strength": final_strength,
            "uncertainty": final_uncertainty,
            "probabilities": final_probabilities,
            "fusion_history": fusion_history,
        }

class ThreeMTAuxiliaryClassifier(nn.Module):
    """
    Produce ordinary class logits from one intermediate
    cumulative 3MT query.

    These outputs support training only and are not interpreted
    as independent modality opinions.
    """

    def __init__(
        self,
        input_dim,
        number_of_classes,
        hidden_dim=64,
        dropout=0.10,
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LeakyReLU(
                negative_slope=0.01,
            ),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                number_of_classes,
            ),
        )

    def forward(self, representation):
        return self.network(
            representation
        )

class ThreeMTPredictionHeads(nn.Module):
    """
    Attach auxiliary classifiers to the intermediate CMT outputs
    and one evidential head to the final 3MT representation.
    """

    def __init__(
        self,
        cascade_order,
        embedding_dim,
        number_of_classes,
    ):
        super().__init__()

        self.cascade_order = list(
            cascade_order
        )

        # The final stage produces the joint evidential opinion.
        self.final_stage = self.cascade_order[-1]

        # Every preceding stage receives an auxiliary classifier.
        self.auxiliary_stages = self.cascade_order[:-1]

        self.auxiliary_heads = nn.ModuleDict(
            {
                stage_name:
                    ThreeMTAuxiliaryClassifier(
                        input_dim=embedding_dim,
                        number_of_classes=number_of_classes,
                        hidden_dim=embedding_dim,
                        dropout=0.10,
                    )

                for stage_name in self.auxiliary_stages
            }
        )

        self.joint_evidential_head = (
            EvidentialClassificationHead(
                input_dim=embedding_dim,
                number_of_classes=number_of_classes,
                hidden_dim=32,
                dropout=0.10,
            )
        )

    def forward(
        self,
        three_mt_output,
    ):
        auxiliary_logits = {}

        # Intermediate auxiliary predictions

        for stage_name in self.auxiliary_stages:

            # Each stored query has shape:
            # (batch_size, 1, embedding_dim).
            stage_representation = three_mt_output[
                "stage_queries"
            ][stage_name].squeeze(1)

            auxiliary_logits[stage_name] = (
                self.auxiliary_heads[
                    stage_name
                ](
                    stage_representation
                )
            )

        # Final joint evidential opinion

        joint_representation = three_mt_output[
            "joint_representation"
        ]

        # The final 3MT query always exists, even when some input
        # modalities are unavailable. I therefore use a branch mask
        # of one for the joint interaction-aware opinion.
        joint_presence_mask = torch.ones(
            joint_representation.shape[0],
            dtype=joint_representation.dtype,
            device=joint_representation.device,
        )

        joint_opinion = self.joint_evidential_head(
            representation=joint_representation,
            branch_mask=joint_presence_mask,
        )

        return {
            "auxiliary_logits":
                auxiliary_logits,

            "joint_opinion":
                joint_opinion,
        }

class FixedEqualHybridFusion(nn.Module):
    """
    Combine calibrated 3MT and TMC evidence with fixed weights.

    The participant-specific reliability gate is removed:

        w_3MT = 0.5
        w_TMC = 0.5

    The two positive pathway evidence scales remain trainable.
    This isolates the contribution of the learned gate without
    changing the remaining fusion architecture.
    """

    def __init__(
        self,
        number_of_classes,
        number_of_modalities,
        initial_three_mt_scale=1.0,
        initial_tmc_scale=1.0,
    ):
        super().__init__()

        self.number_of_classes = number_of_classes
        self.number_of_modalities = number_of_modalities

        self.three_mt_scale_parameter = nn.Parameter(
            inverse_softplus(
                initial_three_mt_scale
            ).clone()
        )

        self.tmc_scale_parameter = nn.Parameter(
            inverse_softplus(
                initial_tmc_scale
            ).clone()
        )

    def _calculate_mean_available_conflict(
        self,
        tmc_output,
        branch_masks,
        modality_order,
    ):
        """
        Calculate the mean TMC conflict across available fusion stages.
        """

        stage_conflicts = []
        stage_masks = []

        for modality_index, modality_name in enumerate(
            modality_order[1:],
            start=1,
        ):
            stage_conflicts.append(
                tmc_output[
                    "fusion_history"
                ][modality_name]["conflict"]
            )

            stage_masks.append(
                branch_masks[
                    :,
                    modality_index,
                ].unsqueeze(-1)
            )

        stacked_conflicts = torch.stack(
            stage_conflicts,
            dim=1,
        )

        stacked_masks = torch.stack(
            stage_masks,
            dim=1,
        )

        conflict_sum = (
            stacked_conflicts
            * stacked_masks
        ).sum(
            dim=1
        )

        available_fusion_count = (
            stacked_masks.sum(
                dim=1
            ).clamp_min(1.0)
        )

        return (
            conflict_sum
            / available_fusion_count
        )

    def forward(
        self,
        joint_opinion,
        tmc_output,
        branch_masks,
        modality_order,
    ):
        three_mt_evidence = joint_opinion[
            "evidence"
        ]

        tmc_evidence = tmc_output[
            "evidence"
        ]

        three_mt_scale = F.softplus(
            self.three_mt_scale_parameter
        )

        tmc_scale = F.softplus(
            self.tmc_scale_parameter
        )

        calibrated_three_mt_evidence = (
            three_mt_scale
            * three_mt_evidence
        )

        calibrated_tmc_evidence = (
            tmc_scale
            * tmc_evidence
        )

        three_mt_uncertainty = joint_opinion[
            "uncertainty"
        ]

        tmc_uncertainty = tmc_output[
            "uncertainty"
        ]

        mean_tmc_conflict = (
            self._calculate_mean_available_conflict(
                tmc_output=tmc_output,
                branch_masks=branch_masks,
                modality_order=modality_order,
            )
        )

        available_modality_count = (
            branch_masks.sum(
                dim=-1,
                keepdim=True,
            )
        )

        available_modality_proportion = (
            available_modality_count
            / float(
                self.number_of_modalities
            )
        )

        three_mt_weight = torch.full_like(
            three_mt_uncertainty,
            fill_value=0.5,
        )

        tmc_weight = torch.full_like(
            tmc_uncertainty,
            fill_value=0.5,
        )

        # These compatibility fields preserve the prediction-table
        # contract used by the learned-gate experiment.
        gate_logit = torch.zeros_like(
            three_mt_weight
        )

        gate_input = torch.cat(
            [
                three_mt_uncertainty.detach(),
                tmc_uncertainty.detach(),
                mean_tmc_conflict.detach(),
                available_modality_proportion,
                branch_masks,
            ],
            dim=-1,
        )

        final_evidence = (
            three_mt_weight
            * calibrated_three_mt_evidence
            +
            tmc_weight
            * calibrated_tmc_evidence
        )

        final_alpha = final_evidence + 1.0

        final_strength = final_alpha.sum(
            dim=-1,
            keepdim=True,
        )

        final_probabilities = (
            final_alpha
            / final_strength
        )

        final_uncertainty = (
            self.number_of_classes
            / final_strength
        )

        return {
            "evidence": final_evidence,
            "alpha": final_alpha,
            "strength": final_strength,
            "probabilities": final_probabilities,
            "uncertainty": final_uncertainty,
            "three_mt_weight": three_mt_weight,
            "tmc_weight": tmc_weight,
            "gate_logit": gate_logit,
            "gate_input": gate_input,
            "mean_tmc_conflict": mean_tmc_conflict,
            "available_modality_count": available_modality_count,
            "three_mt_scale": three_mt_scale,
            "tmc_scale": tmc_scale,
            "calibrated_three_mt_evidence":
                calibrated_three_mt_evidence,
            "calibrated_tmc_evidence":
                calibrated_tmc_evidence,
        }

class ADNIEvidential3MTTMCModel(nn.Module):
    """
    Complete missing-aware and uncertainty-aware multimodal model.

    The model combines:

    1. six modality-specific encoders;
    2. training-time modality dropout;
    3. independent modality evidential heads;
    4. TMC/Dempster-Shafer fusion;
    5. the availability-gated 3MT interaction pathway;
    6. intermediate 3MT auxiliary classifiers;
    7. a joint 3MT evidential head;
    8. fixed-equal final evidence fusion.
    """

    def __init__(
        self,
        embedding_dim=MODALITY_EMBEDDING_DIM,
        number_of_classes=NUMBER_OF_CLASSES,
        modality_order=MODALITY_ORDER,
        cascade_order=THREE_MT_CASCADE_ORDER,
        modality_dropout_probability=0.50,
    ):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.number_of_classes = number_of_classes

        self.modality_order = list(
            modality_order
        )

        self.cascade_order = list(
            cascade_order
        )

        self.number_of_modalities = len(
            self.modality_order
        )

        self.modality_dropout_probability = (
            modality_dropout_probability
        )

        # Modality-specific encoders

        # This wrapper returns both raw and branch-masked modality
        # representations.
        self.modality_encoders = (
            MaskedADNIModalityEncoders(
                output_dim=embedding_dim,
            )
        )

        # Independent modality evidential pathway

        self.independent_evidential_heads = (
            IndependentModalityEvidentialHeads(
                modality_order=self.modality_order,
                input_dim=embedding_dim,
                number_of_classes=number_of_classes,
            )
        )

        self.tmc_fusion = TMCFusion(
            number_of_classes=number_of_classes,
            modality_order=self.modality_order,
        )

        # Interaction-aware 3MT pathway

        self.three_mt_cascade = ThreeMTCascade(
            embedding_dim=embedding_dim,
            modality_order=self.modality_order,
            cascade_order=self.cascade_order,
            number_of_heads=4,
            dropout=0.10,
        )

        self.three_mt_prediction_heads = (
            ThreeMTPredictionHeads(
                cascade_order=self.cascade_order,
                embedding_dim=embedding_dim,
                number_of_classes=number_of_classes,
            )
        )

        # Final fixed 50/50 hybrid fusion

        self.hybrid_fusion = (
            FixedEqualHybridFusion(
                number_of_classes=number_of_classes,
                number_of_modalities=self.number_of_modalities,
                initial_three_mt_scale=1.0,
                initial_tmc_scale=1.0,
            )
        )

    # Training-time modality dropout

    def _apply_modality_dropout(
        self,
        original_branch_masks,
    ):
        """
        Randomly hide genuinely available modalities during training.

        Parameters
        ----------
        original_branch_masks:
            Tensor of shape:
            (batch_size, number_of_modalities)

        Returns
        -------
        effective_branch_masks:
            Masks after training-time modality dropout.

        dropped_branch_masks:
            Indicators showing which originally available branches
            were hidden by modality dropout.
        """

        # Validation and testing always use the genuine prepared
        # availability pattern.
        if (
            not self.training
            or self.modality_dropout_probability <= 0.0
        ):
            effective_branch_masks = (
                original_branch_masks.clone()
            )

            dropped_branch_masks = torch.zeros_like(
                original_branch_masks
            )

            return (
                effective_branch_masks,
                dropped_branch_masks,
            )

        # Sample branch-retention indicators

        retention_probability = (
            1.0
            - self.modality_dropout_probability
        )

        retention_masks = torch.bernoulli(
            torch.full_like(
                original_branch_masks,
                fill_value=retention_probability,
            )
        )

        # A naturally unavailable modality remains unavailable.
        effective_branch_masks = (
            original_branch_masks
            * retention_masks
        )

        # Prevent complete information removal

        batch_size = original_branch_masks.shape[0]

        for batch_row in range(batch_size):

            originally_available_indices = torch.nonzero(
                original_branch_masks[
                    batch_row
                ] > 0,
                as_tuple=False,
            ).flatten()

            no_effective_modality = (
                effective_branch_masks[
                    batch_row
                ].sum()
                == 0
            )

            if (
                no_effective_modality
                and originally_available_indices.numel() > 0
            ):
                # Restore one branch that was genuinely
                # available for this participant.
                selected_position = torch.randint(
                    low=0,
                    high=originally_available_indices.numel(),
                    size=(1,),
                    device=original_branch_masks.device,
                )

                selected_modality_index = (
                    originally_available_indices[
                        selected_position
                    ].item()
                )

                effective_branch_masks[
                    batch_row,
                    selected_modality_index,
                ] = 1.0

        dropped_branch_masks = (
            original_branch_masks
            - effective_branch_masks
        ).clamp(
            min=0.0,
            max=1.0,
        )

        return (
            effective_branch_masks,
            dropped_branch_masks,
        )

    # Construct effective modality dictionaries

    def _replace_branch_masks(
        self,
        modalities,
        effective_branch_masks,
    ):
        """
        Construct a new modality dictionary containing the
        training-time effective branch masks.
        """

        effective_modalities = {}

        for modality_index, modality_name in enumerate(
            self.modality_order
        ):
            effective_modalities[modality_name] = dict(
                modalities[modality_name]
            )

            effective_modalities[
                modality_name
            ]["branch_mask"] = (
                effective_branch_masks[
                    :,
                    modality_index,
                ]
            )

        return effective_modalities

    # Complete forward pass

    def forward(
        self,
        modalities,
        original_branch_masks,
    ):
        """
        Run the complete multimodal architecture.

        Parameters
        ----------
        modalities:
            Nested modality dictionary produced by the dataset.

        original_branch_masks:
            Genuine prepared modality-availability tensor with shape:
            (batch_size, number_of_modalities).
        """

        # Apply training-time modality dropout

        (
            effective_branch_masks,
            dropped_branch_masks,
        ) = self._apply_modality_dropout(
            original_branch_masks
        )

        effective_modalities = (
            self._replace_branch_masks(
                modalities=modalities,
                effective_branch_masks=effective_branch_masks,
            )
        )

        # Encode all six modalities

        encoded_modalities = self.modality_encoders(
            effective_modalities
        )

        # The encoders have already applied their effective branch
        # masks. Use these representations for both pathways.
        masked_representations = encoded_modalities[
            "masked"
        ]

        # Independent modality opinions and TMC fusion

        modality_opinions = (
            self.independent_evidential_heads(
                modality_representations=masked_representations,
                modalities=effective_modalities,
            )
        )

        tmc_output = self.tmc_fusion(
            modality_opinions=modality_opinions
        )

        # Interaction-aware 3MT pathway

        three_mt_output = self.three_mt_cascade(
            masked_representations=masked_representations,
            branch_masks=effective_branch_masks,
        )

        three_mt_predictions = (
            self.three_mt_prediction_heads(
                three_mt_output=three_mt_output
            )
        )

        joint_opinion = three_mt_predictions[
            "joint_opinion"
        ]

        # Final fixed-equal hybrid opinion

        final_output = self.hybrid_fusion(
            joint_opinion=joint_opinion,
            tmc_output=tmc_output,
            branch_masks=effective_branch_masks,
            modality_order=self.modality_order,
        )

        return {
            # Final main prediction
            "final_output":
                final_output,

            # Independent uncertainty pathway
            "modality_opinions":
                modality_opinions,

            "tmc_output":
                tmc_output,

            # Interaction-aware pathway
            "three_mt_output":
                three_mt_output,

            "three_mt_predictions":
                three_mt_predictions,

            "joint_opinion":
                joint_opinion,

            # Encoder outputs
            "encoded_modalities":
                encoded_modalities,

            # Missingness and training-time dropout information
            "original_branch_masks":
                original_branch_masks,

            "effective_branch_masks":
                effective_branch_masks,

            "dropped_branch_masks":
                dropped_branch_masks,
        }


In [ ]:

# 4. Loading folds/checkpoints and MC-dropout helpers

BRANCH_MASK_COLUMNS = [
    "BRANCH_MASK__DEMOGRAPHICS",
    "BRANCH_MASK__COGNITIVE_FUNCTIONAL",
    "BRANCH_MASK__CSF",
    "BRANCH_MASK__PLASMA",
    "BRANCH_MASK__APOE",
    "BRANCH_MASK__MRI",
]

BRANCH_DISPLAY_NAMES = {
    "demographics": "Demographics",
    "cognitive_functional": "Cognitive/Functional",
    "csf": "CSF",
    "plasma": "Plasma",
    "apoe": "APOE",
    "mri": "MRI",
}

def set_global_seed(seed=20260928):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def move_nested_to_device(obj, device):
    if torch.is_tensor(obj):
        return obj.to(device)
    if isinstance(obj, dict):
        return {k: move_nested_to_device(v, device) for k, v in obj.items()}
    if isinstance(obj, list):
        return [move_nested_to_device(v, device) for v in obj]
    if isinstance(obj, tuple):
        return tuple(move_nested_to_device(v, device) for v in obj)
    return obj

def fold_input_path(fold):
    return INPUT_ROOT / f"mci_prognosis_outer_fold_{fold}_final_task_ready.csv"

def checkpoint_path(fold):
    return (
        EXPERIMENT_ROOT
        / f"fold_{fold}"
        / "checkpoints"
        / "best_validation_auc_checkpoint.pt"
    )

def load_schema():
    with open(SCHEMA_PATH, "r") as f:
        return json.load(f)

def make_test_loader(fold):
    path = fold_input_path(fold)
    if not path.exists():
        raise FileNotFoundError(path)

    table = pd.read_csv(path)
    test_table = table.loc[table["DATA_ROLE"].astype(str).str.lower() == "test"].copy()

    if len(test_table) == 0:
        raise ValueError(f"No test rows found for fold {fold}")

    dataset = ADNIMultimodalDataset(
        participant_table=test_table,
        dataset_column_contract=dataset_column_contract,
        mri_tensor_shape=MRI_TENSOR_SHAPE,
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    return test_table, loader

def build_and_load_model(fold):
    model = ADNIEvidential3MTTMCModel(
        modality_embedding_dim=MODALITY_EMBEDDING_DIM,
        number_of_classes=NUMBER_OF_CLASSES,
        modality_dropout_probability=0.50,
    ).to(DEVICE)

    ckpt_path = checkpoint_path(fold)
    if not ckpt_path.exists():
        raise FileNotFoundError(ckpt_path)

    checkpoint = torch.load(
        ckpt_path,
        map_location=DEVICE,
        weights_only=False,
    )

    # Support common checkpoint layouts.
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        state_dict = checkpoint

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing or unexpected:
        raise RuntimeError(
            f"Checkpoint mismatch for fold {fold}. "
            f"Missing={missing}, unexpected={unexpected}"
        )

    model.eval()
    return model

def disable_all_stochastic_dropout(model):
    """Return the complete model to deterministic evaluation mode."""
    model.eval()

def enable_mc_dropout_only(model):
    """
    Keep the TOP-LEVEL model in eval mode, while selectively switching only
    neural dropout-bearing submodules to training mode.

    This is the key safety rule for this project:
      - standard neural dropout ON  -> stochastic MC samples
      - model.training remains False -> 50% modality dropout stays OFF
      - natural clinical branch masks remain unchanged
    """
    model.eval()

    # TransformerEncoderLayer must be set to train as well; otherwise some
    # PyTorch versions may use an eval fast path that bypasses child dropout.
    stochastic_types = (
        nn.Dropout,
        nn.Dropout1d,
        nn.Dropout2d,
        nn.Dropout3d,
        nn.MultiheadAttention,
        nn.TransformerEncoderLayer,
    )

    count = 0
    for module in model.modules():
        if isinstance(module, stochastic_types):
            module.train()
            count += 1

    if model.training:
        raise RuntimeError(
            "Top-level model unexpectedly entered training mode. "
            "That would reactivate modality dropout."
        )
    return count

def binary_entropy_from_prob(p, eps=1e-8):
    p = np.clip(np.asarray(p, dtype=float), eps, 1.0 - eps)
    return -(p * np.log(p) + (1.0 - p) * np.log(1.0 - p))

def multiclass_entropy(prob, eps=1e-8):
    prob = np.clip(np.asarray(prob, dtype=float), eps, 1.0)
    return -np.sum(prob * np.log(prob), axis=-1)

def max_pairwise_conflict_from_output(model_output, branch_masks):
    """
    Compute maximum pairwise conflict from the independent modality beliefs.

    For binary opinions a,b:
        C(a,b) = b_a0*b_b1 + b_a1*b_b0

    Only modality pairs genuinely available for each participant are eligible.
    """
    beliefs = model_output["modality_opinions"]
    batch_size = branch_masks.shape[0]

    pair_conflicts = []
    pair_available = []

    for i, j in combinations(range(len(MODALITY_ORDER)), 2):
        mi = MODALITY_ORDER[i]
        mj = MODALITY_ORDER[j]

        bi = beliefs[mi]["belief"]
        bj = beliefs[mj]["belief"]

        conflict = bi[:, 0] * bj[:, 1] + bi[:, 1] * bj[:, 0]
        available = (branch_masks[:, i] > 0.5) & (branch_masks[:, j] > 0.5)

        pair_conflicts.append(conflict)
        pair_available.append(available)

    C = torch.stack(pair_conflicts, dim=1)
    A = torch.stack(pair_available, dim=1)

    C = torch.where(A, C, torch.full_like(C, -1.0))
    cmax = C.max(dim=1).values
    cmax = torch.where(cmax >= 0, cmax, torch.zeros_like(cmax))
    return cmax

def unpack_ids(batch, batch_size):
    # PyTorch collates string IDs as lists/tuples. Fall back to RID tensor.
    if "rid" in batch:
        rid = batch["rid"]
        if torch.is_tensor(rid):
            rid = rid.detach().cpu().numpy().tolist()
        else:
            rid = list(rid)
    else:
        rid = [None] * batch_size

    # Optional participant string ID, if present.
    ptid = batch.get("ptid", [None] * batch_size)
    if torch.is_tensor(ptid):
        ptid = ptid.detach().cpu().numpy().tolist()
    elif isinstance(ptid, (list, tuple)):
        ptid = list(ptid)
    else:
        ptid = [ptid] * batch_size

    return rid, ptid


## Main experiment

For every held-out participant the notebook stores the deterministic prediction, the current evidential uncertainty, maximum pairwise TMC conflict, the current conflict-aware uncertainty, and MC-dropout uncertainty derived from repeated stochastic forward passes.


In [ ]:

# 5. Deterministic + Monte Carlo dropout inference

def run_fold_mc_dropout(fold, mc_samples=MC_SAMPLES):
    test_table, loader = make_test_loader(fold)
    model = build_and_load_model(fold)

    fold_rows = []

    for batch_index, batch_cpu in enumerate(loader):
        batch = move_nested_to_device(batch_cpu, DEVICE)

        target = batch["target"].long()
        branch_masks = batch["branch_masks"].float()
        modalities = batch["modalities"]

        batch_size = target.shape[0]
        rid, ptid = unpack_ids(batch_cpu, batch_size)

        # A. Deterministic forward pass
        disable_all_stochastic_dropout(model)
        with torch.no_grad():
            det_output = model(
                modalities=modalities,
                original_branch_masks=branch_masks,
            )

        det_prob = det_output["final_output"]["probabilities"]
        det_u = det_output["final_output"]["uncertainty"].reshape(-1)
        det_cmax = max_pairwise_conflict_from_output(det_output, branch_masks)
        det_u_conflict = 1.0 - (1.0 - det_u) * (1.0 - det_cmax)

        # B. MC-dropout stochastic passes
        dropout_module_count = enable_mc_dropout_only(model)
        if batch_index == 0:
            print(
                f"Fold {fold}: activated {dropout_module_count} stochastic "
                "dropout/attention modules while keeping top-level eval mode."
            )

        sample_probs = []
        sample_evidential_u = []

        with torch.no_grad():
            for _ in range(mc_samples):
                out = model(
                    modalities=modalities,
                    original_branch_masks=branch_masks,
                )
                sample_probs.append(
                    out["final_output"]["probabilities"].detach().cpu().numpy()
                )
                sample_evidential_u.append(
                    out["final_output"]["uncertainty"].reshape(-1).detach().cpu().numpy()
                )

        probs = np.stack(sample_probs, axis=0)  # [T, B, 2]
        u_samples = np.stack(sample_evidential_u, axis=0)  # [T, B]

        mean_prob = probs.mean(axis=0)
        p_pmci_samples = probs[:, :, 1]

        mc_std = p_pmci_samples.std(axis=0, ddof=1 if mc_samples > 1 else 0)
        pred_entropy = multiclass_entropy(mean_prob)
        expected_entropy = multiclass_entropy(probs).mean(axis=0)
        mutual_information = pred_entropy - expected_entropy

        target_np = target.detach().cpu().numpy()
        branch_np = branch_masks.detach().cpu().numpy()
        det_prob_np = det_prob.detach().cpu().numpy()
        det_u_np = det_u.detach().cpu().numpy()
        det_cmax_np = det_cmax.detach().cpu().numpy()
        det_u_conflict_np = det_u_conflict.detach().cpu().numpy()

        for i in range(batch_size):
            det_p = float(det_prob_np[i, 1])
            mc_p = float(mean_prob[i, 1])
            y = int(target_np[i])

            det_pred = int(det_p >= CLASSIFICATION_THRESHOLD)
            mc_pred = int(mc_p >= CLASSIFICATION_THRESHOLD)

            row = {
                "OUTER_FOLD": fold,
                "RID": rid[i],
                "PTID": ptid[i],
                "TARGET": y,
                "MODALITY_COUNT": int(branch_np[i].sum()),
                "DET_P_PMCI": det_p,
                "DET_PRED": det_pred,
                "ERROR_DETERMINISTIC": int(det_pred != y),
                "DET_EVIDENTIAL_UNCERTAINTY": float(det_u_np[i]),
                "DET_MAX_PAIRWISE_CONFLICT": float(det_cmax_np[i]),
                "DET_CONFLICT_AWARE_UNCERTAINTY": float(det_u_conflict_np[i]),
                "MC_MEAN_P_PMCI": mc_p,
                "MC_PRED": mc_pred,
                "ERROR_MC_MEAN": int(mc_pred != y),
                "MC_P_PMCI_STD": float(mc_std[i]),
                "MC_PREDICTIVE_ENTROPY": float(pred_entropy[i]),
                "MC_EXPECTED_ENTROPY": float(expected_entropy[i]),
                "MC_MUTUAL_INFORMATION": float(mutual_information[i]),
                "MC_MEAN_EVIDENTIAL_UNCERTAINTY": float(u_samples[:, i].mean()),
            }

            for j, name in enumerate(MODALITY_ORDER):
                row[f"AVAILABLE__{name.upper()}"] = int(branch_np[i, j] > 0.5)

            fold_rows.append(row)

    disable_all_stochastic_dropout(model)
    return pd.DataFrame(fold_rows)

set_global_seed()

all_fold_predictions = []
for fold in OUTER_FOLDS:
    print(f"\nRunning fold {fold} ...")
    fold_df = run_fold_mc_dropout(fold, mc_samples=MC_SAMPLES)
    all_fold_predictions.append(fold_df)
    print(f"  completed {len(fold_df)} held-out participants")

mc_oof = pd.concat(all_fold_predictions, ignore_index=True)

output_name = (
    "pilot_fold0_mc_dropout_predictions.csv"
    if PILOT_MODE
    else "pooled_five_fold_mc_dropout_predictions.csv"
)
mc_oof.to_csv(TABLE_DIR / output_name, index=False)

display(mc_oof.head())
print("\nRows:", len(mc_oof))
print("Unique RID:", mc_oof["RID"].nunique())
print("Saved:", TABLE_DIR / output_name)


In [ ]:

# 6. Apples-to-apples uncertainty comparison
#
# The primary comparison below uses ERROR_DETERMINISTIC for EVERY uncertainty
# score. This isolates the question "which uncertainty score best identifies
# errors made by the same fixed deterministic classifier?"
#
# MC-mean prediction performance is reported separately, because MC averaging
# can slightly change the prediction itself.

UNCERTAINTY_SCORES = {
    "Evidential u=K/S": "DET_EVIDENTIAL_UNCERTAINTY",
    "Conflict-aware evidential": "DET_CONFLICT_AWARE_UNCERTAINTY",
    "MC predictive entropy": "MC_PREDICTIVE_ENTROPY",
    "MC mutual information": "MC_MUTUAL_INFORMATION",
    "MC probability std": "MC_P_PMCI_STD",
}

def safe_auc(y, score):
    y = np.asarray(y)
    score = np.asarray(score)
    if len(np.unique(y)) < 2:
        return np.nan
    return roc_auc_score(y, score)

def uncertainty_error_detection_table(df, error_col="ERROR_DETERMINISTIC"):
    rows = []
    y_error = df[error_col].to_numpy()

    for label, col in UNCERTAINTY_SCORES.items():
        score = df[col].to_numpy()
        rows.append({
            "uncertainty_method": label,
            "score_column": col,
            "error_definition": error_col,
            "error_detection_roc_auc": safe_auc(y_error, score),
            "error_detection_average_precision": average_precision_score(y_error, score),
            "mean_correct": float(df.loc[df[error_col] == 0, col].mean()),
            "mean_incorrect": float(df.loc[df[error_col] == 1, col].mean()),
            "incorrect_minus_correct": float(
                df.loc[df[error_col] == 1, col].mean()
                - df.loc[df[error_col] == 0, col].mean()
            ),
        })
    return pd.DataFrame(rows)

error_detection = uncertainty_error_detection_table(mc_oof)
display(error_detection)

error_detection.to_csv(
    TABLE_DIR / (
        "pilot_uncertainty_error_detection.csv"
        if PILOT_MODE else "uncertainty_error_detection.csv"
    ),
    index=False,
)

# Prediction/calibration comparison: deterministic probability vs MC-averaged probability.
def prediction_quality(df, prob_col):
    y = df["TARGET"].to_numpy()
    p = np.clip(df[prob_col].to_numpy(), 1e-8, 1 - 1e-8)
    pred = (p >= CLASSIFICATION_THRESHOLD).astype(int)

    return {
        "probability": prob_col,
        "roc_auc": roc_auc_score(y, p),
        "average_precision": average_precision_score(y, p),
        "brier": brier_score_loss(y, p),
        "nll": log_loss(y, np.column_stack([1-p, p]), labels=[0, 1]),
        "error_rate": float(np.mean(pred != y)),
        "prediction_flips_vs_deterministic": (
            np.nan if prob_col == "DET_P_PMCI"
            else int(np.sum(pred != (df["DET_P_PMCI"].to_numpy() >= 0.5)))
        ),
    }

prediction_summary = pd.DataFrame([
    prediction_quality(mc_oof, "DET_P_PMCI"),
    prediction_quality(mc_oof, "MC_MEAN_P_PMCI"),
])
display(prediction_summary)

prediction_summary.to_csv(
    TABLE_DIR / (
        "pilot_prediction_quality.csv"
        if PILOT_MODE else "prediction_quality.csv"
    ),
    index=False,
)

if not PILOT_MODE:
    evid_auc = error_detection.loc[
        error_detection["uncertainty_method"] == "Evidential u=K/S",
        "error_detection_roc_auc"
    ].iloc[0]
    conf_auc = error_detection.loc[
        error_detection["uncertainty_method"] == "Conflict-aware evidential",
        "error_detection_roc_auc"
    ].iloc[0]

    print("\nDeterministic reconstruction check:")
    print(
        f"  Evidential error AUC: {evid_auc:.4f} "
        f"(dissertation reference {REFERENCE_EVIDENTIAL_ERROR_AUC:.4f})"
    )
    print(
        f"  Conflict-aware error AUC: {conf_auc:.4f} "
        f"(dissertation reference {REFERENCE_CONFLICT_ERROR_AUC:.4f})"
    )

    if abs(evid_auc - REFERENCE_EVIDENTIAL_ERROR_AUC) > 0.02:
        warnings.warn(
            "The reconstructed evidential score differs materially from the "
            "dissertation result. Check checkpoint/input versions before "
            "interpreting the MC comparison."
        )


In [ ]:

# 7. Paired participant-level bootstrap comparisons

def paired_bootstrap_score_difference(
    df,
    score_a,
    score_b,
    metric="roc_auc",
    error_col="ERROR_DETERMINISTIC",
    iterations=BOOTSTRAP_ITERATIONS,
    seed=BOOTSTRAP_SEED,
):
    rng = np.random.default_rng(seed)
    y = df[error_col].to_numpy()
    a = df[score_a].to_numpy()
    b = df[score_b].to_numpy()
    n = len(df)

    if metric == "roc_auc":
        metric_fn = roc_auc_score
    elif metric == "average_precision":
        metric_fn = average_precision_score
    else:
        raise ValueError(metric)

    observed = metric_fn(y, a) - metric_fn(y, b)
    diffs = []

    for _ in range(iterations):
        idx = rng.integers(0, n, size=n)
        yb = y[idx]
        if len(np.unique(yb)) < 2:
            continue
        diffs.append(metric_fn(yb, a[idx]) - metric_fn(yb, b[idx]))

    diffs = np.asarray(diffs)
    return {
        "score_a": score_a,
        "score_b": score_b,
        "metric": metric,
        "observed_difference_a_minus_b": observed,
        "ci_2.5": np.percentile(diffs, 2.5),
        "ci_97.5": np.percentile(diffs, 97.5),
        "bootstrap_n": len(diffs),
    }

comparison_rows = []
for mc_score in [
    "MC_PREDICTIVE_ENTROPY",
    "MC_MUTUAL_INFORMATION",
    "MC_P_PMCI_STD",
]:
    for baseline in [
        "DET_EVIDENTIAL_UNCERTAINTY",
        "DET_CONFLICT_AWARE_UNCERTAINTY",
    ]:
        comparison_rows.append(
            paired_bootstrap_score_difference(
                mc_oof,
                mc_score,
                baseline,
                metric="roc_auc",
            )
        )
        comparison_rows.append(
            paired_bootstrap_score_difference(
                mc_oof,
                mc_score,
                baseline,
                metric="average_precision",
            )
        )

paired_bootstrap_comparisons = pd.DataFrame(comparison_rows)
display(paired_bootstrap_comparisons)

paired_bootstrap_comparisons.to_csv(
    TABLE_DIR / (
        "pilot_paired_bootstrap_uncertainty.csv"
        if PILOT_MODE else "paired_bootstrap_uncertainty.csv"
    ),
    index=False,
)


In [ ]:

# 8. Risk-coverage, modality availability, and score correlations

def risk_coverage_table(df, score_col, error_col="ERROR_DETERMINISTIC"):
    rows = []
    ordered = df.sort_values(score_col, ascending=True).reset_index(drop=True)

    for coverage in np.arange(1.0, 0.49, -0.05):
        keep_n = max(1, int(np.floor(len(ordered) * coverage)))
        kept = ordered.iloc[:keep_n]

        y = kept["TARGET"].to_numpy()
        p = kept["DET_P_PMCI"].to_numpy()

        rows.append({
            "score": score_col,
            "coverage": float(coverage),
            "n_retained": len(kept),
            "error_rate": float(kept[error_col].mean()),
            "retained_roc_auc": (
                roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan
            ),
        })

    return pd.DataFrame(rows)

risk_tables = []
for _, col in UNCERTAINTY_SCORES.items():
    risk_tables.append(risk_coverage_table(mc_oof, col))
risk_coverage = pd.concat(risk_tables, ignore_index=True)

risk_coverage.to_csv(
    TABLE_DIR / (
        "pilot_risk_coverage.csv"
        if PILOT_MODE else "risk_coverage.csv"
    ),
    index=False,
)

# Association with number of available modalities.
availability_rows = []
for label, col in UNCERTAINTY_SCORES.items():
    rho, pval = spearmanr(mc_oof["MODALITY_COUNT"], mc_oof[col])
    availability_rows.append({
        "uncertainty_method": label,
        "score_column": col,
        "spearman_rho_modality_count": rho,
        "p_value": pval,
    })

availability_association = pd.DataFrame(availability_rows)
display(availability_association)

# Rank/linear relationships between uncertainty methods.
score_cols = list(UNCERTAINTY_SCORES.values())
spearman_matrix = mc_oof[score_cols].corr(method="spearman")
display(spearman_matrix)

availability_association.to_csv(
    TABLE_DIR / (
        "pilot_uncertainty_vs_modality_count.csv"
        if PILOT_MODE else "uncertainty_vs_modality_count.csv"
    ),
    index=False,
)
spearman_matrix.to_csv(
    TABLE_DIR / (
        "pilot_uncertainty_score_spearman_matrix.csv"
        if PILOT_MODE else "uncertainty_score_spearman_matrix.csv"
    )
)

# Plot risk-coverage on one figure.
fig, ax = plt.subplots(figsize=(8, 5))
for label, col in UNCERTAINTY_SCORES.items():
    tmp = risk_coverage[risk_coverage["score"] == col]
    ax.plot(tmp["coverage"], tmp["error_rate"], marker="o", label=label)

ax.set_xlabel("Coverage (fraction retained)")
ax.set_ylabel("Error rate among retained participants")
ax.set_title("Risk-coverage comparison")
ax.invert_xaxis()
ax.legend(fontsize=8)
fig.tight_layout()

figure_name = (
    "pilot_risk_coverage_comparison.png"
    if PILOT_MODE else "risk_coverage_comparison.png"
)
fig.savefig(FIGURE_DIR / figure_name, dpi=180)
plt.show()


In [ ]:

# 9. Error rates by uncertainty quintile

quintile_rows = []

for label, col in UNCERTAINTY_SCORES.items():
    temp = mc_oof[["ERROR_DETERMINISTIC", col]].copy()
    # duplicates='drop' protects against a degenerate pilot distribution.
    temp["quintile"] = pd.qcut(
        temp[col].rank(method="first"),
        q=5,
        labels=False,
        duplicates="drop",
    ) + 1

    for q, g in temp.groupby("quintile"):
        quintile_rows.append({
            "uncertainty_method": label,
            "score_column": col,
            "quintile": int(q),
            "n": len(g),
            "mean_uncertainty": float(g[col].mean()),
            "error_rate": float(g["ERROR_DETERMINISTIC"].mean()),
        })

quintile_table = pd.DataFrame(quintile_rows)
display(quintile_table)

quintile_table.to_csv(
    TABLE_DIR / (
        "pilot_error_rate_by_uncertainty_quintile.csv"
        if PILOT_MODE else "error_rate_by_uncertainty_quintile.csv"
    ),
    index=False,
)


## Optional uncertainty attribution

The next cell implements leave-one-branch-out MC-dropout attribution. It is disabled by default because it is substantially more expensive.


In [ ]:

# 10. OPTIONAL: leave-one-branch-out MC-dropout attribution
#
# This answers a different question from the existing leave-one-opinion-out
# analysis:
#
#   Existing attribution:
#       remove one TMC opinion from the fitted evidence fusion.
#
#   This LOBO attribution:
#       remove one observed branch from the WHOLE hybrid computation, so it
#       can affect the 3MT interaction path and the TMC path.
#
# It is a perturbation/sensitivity analysis, not a causal estimate of clinical
# modality importance.

def seeded_mc_metrics(
    model,
    modalities,
    branch_masks,
    mc_samples,
    sample_seeds,
):
    enable_mc_dropout_only(model)

    probs = []
    with torch.no_grad():
        for seed in sample_seeds:
            torch.manual_seed(int(seed))
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(int(seed))

            out = model(
                modalities=modalities,
                original_branch_masks=branch_masks,
            )
            probs.append(
                out["final_output"]["probabilities"].detach().cpu().numpy()
            )

    probs = np.stack(probs, axis=0)
    mean_prob = probs.mean(axis=0)
    pred_entropy = multiclass_entropy(mean_prob)
    expected_entropy = multiclass_entropy(probs).mean(axis=0)
    mi = pred_entropy - expected_entropy
    std = probs[:, :, 1].std(
        axis=0,
        ddof=1 if mc_samples > 1 else 0,
    )

    return {
        "mean_p_pmci": mean_prob[:, 1],
        "predictive_entropy": pred_entropy,
        "mutual_information": mi,
        "p_std": std,
    }

def run_fold_lobo_attribution(fold, mc_samples=LOBO_MC_SAMPLES):
    _, loader = make_test_loader(fold)
    model = build_and_load_model(fold)

    rows = []
    global_batch_counter = 0

    for batch_cpu in loader:
        batch = move_nested_to_device(batch_cpu, DEVICE)
        target = batch["target"].long()
        base_masks = batch["branch_masks"].float()
        modalities = batch["modalities"]

        batch_size = target.shape[0]
        rid, ptid = unpack_ids(batch_cpu, batch_size)
        target_np = target.detach().cpu().numpy()
        base_masks_np = base_masks.detach().cpu().numpy()

        # Common random numbers: use the same dropout seeds for the full model
        # and every branch ablation in this batch. This reduces Monte Carlo noise
        # in the delta attributed to a branch.
        sample_seeds = [
            900000 + fold * 10000 + global_batch_counter * 100 + t
            for t in range(mc_samples)
        ]

        full = seeded_mc_metrics(
            model, modalities, base_masks, mc_samples, sample_seeds
        )

        for m_idx, modality in enumerate(MODALITY_ORDER):
            ablated_masks = base_masks.clone()
            ablated_masks[:, m_idx] = 0.0

            ablated = seeded_mc_metrics(
                model, modalities, ablated_masks, mc_samples, sample_seeds
            )

            for i in range(batch_size):
                was_available = bool(base_masks_np[i, m_idx] > 0.5)

                rows.append({
                    "OUTER_FOLD": fold,
                    "RID": rid[i],
                    "PTID": ptid[i],
                    "TARGET": int(target_np[i]),
                    "MODALITY": modality,
                    "MODALITY_DISPLAY": BRANCH_DISPLAY_NAMES[modality],
                    "WAS_NATURALLY_AVAILABLE": int(was_available),

                    "FULL_MC_MEAN_P_PMCI": float(full["mean_p_pmci"][i]),
                    "ABL_MC_MEAN_P_PMCI": (
                        float(ablated["mean_p_pmci"][i])
                        if was_available else np.nan
                    ),
                    "ABS_DELTA_P_PMCI": (
                        abs(float(ablated["mean_p_pmci"][i] - full["mean_p_pmci"][i]))
                        if was_available else np.nan
                    ),

                    "FULL_MC_PREDICTIVE_ENTROPY": float(full["predictive_entropy"][i]),
                    "ABL_MC_PREDICTIVE_ENTROPY": (
                        float(ablated["predictive_entropy"][i])
                        if was_available else np.nan
                    ),
                    "DELTA_PREDICTIVE_ENTROPY": (
                        float(ablated["predictive_entropy"][i] - full["predictive_entropy"][i])
                        if was_available else np.nan
                    ),

                    "FULL_MC_MUTUAL_INFORMATION": float(full["mutual_information"][i]),
                    "ABL_MC_MUTUAL_INFORMATION": (
                        float(ablated["mutual_information"][i])
                        if was_available else np.nan
                    ),
                    "DELTA_MUTUAL_INFORMATION": (
                        float(ablated["mutual_information"][i] - full["mutual_information"][i])
                        if was_available else np.nan
                    ),

                    "FULL_MC_P_STD": float(full["p_std"][i]),
                    "ABL_MC_P_STD": (
                        float(ablated["p_std"][i])
                        if was_available else np.nan
                    ),
                    "DELTA_P_STD": (
                        float(ablated["p_std"][i] - full["p_std"][i])
                        if was_available else np.nan
                    ),
                })

        global_batch_counter += 1

    disable_all_stochastic_dropout(model)
    return pd.DataFrame(rows)

if RUN_LOBO_ATTRIBUTION:
    lobo_frames = []
    for fold in OUTER_FOLDS:
        print(f"Running LOBO attribution for fold {fold} ...")
        lobo_frames.append(
            run_fold_lobo_attribution(
                fold,
                mc_samples=LOBO_MC_SAMPLES,
            )
        )

    lobo_participant = pd.concat(lobo_frames, ignore_index=True)

    lobo_participant.to_csv(
        TABLE_DIR / (
            "pilot_lobo_mc_dropout_participant_level.csv"
            if PILOT_MODE else "lobo_mc_dropout_participant_level.csv"
        ),
        index=False,
    )

    available_lobo = lobo_participant[
        lobo_participant["WAS_NATURALLY_AVAILABLE"] == 1
    ].copy()

    lobo_summary = (
        available_lobo
        .groupby(["MODALITY", "MODALITY_DISPLAY"], as_index=False)
        .agg(
            n=("RID", "count"),
            mean_abs_delta_probability=("ABS_DELTA_P_PMCI", "mean"),
            median_abs_delta_probability=("ABS_DELTA_P_PMCI", "median"),
            mean_delta_predictive_entropy=("DELTA_PREDICTIVE_ENTROPY", "mean"),
            mean_delta_mutual_information=("DELTA_MUTUAL_INFORMATION", "mean"),
            median_delta_mutual_information=("DELTA_MUTUAL_INFORMATION", "median"),
            mean_delta_probability_std=("DELTA_P_STD", "mean"),
        )
        .sort_values("mean_abs_delta_probability", ascending=False)
    )

    lobo_summary.to_csv(
        TABLE_DIR / (
            "pilot_lobo_mc_dropout_modality_summary.csv"
            if PILOT_MODE else "lobo_mc_dropout_modality_summary.csv"
        ),
        index=False,
    )

    display(lobo_summary)
else:
    print(
        "LOBO attribution is OFF. Set RUN_LOBO_ATTRIBUTION=True after the "
        "main MC-dropout run is validated."
    )


## 11. How to interpret the comparison

Use the **same held-out participants and the same deterministic errors** for the primary uncertainty comparison.

The dissertation's current scores answer:

- **Evidential `u = K/S`**: how little class evidence the network produced.
- **Conflict-aware evidential uncertainty**: how little evidence was produced **plus** whether available modality opinions disagree.

The MC-dropout scores answer a different question:

- **MC probability standard deviation**: how much the pMCI probability changes across stochastic subnetworks.
- **Predictive entropy**: total ambiguity of the MC-averaged predictive distribution.
- **Mutual information**: the part of predictive uncertainty associated with disagreement across the stochastic model samples, and therefore the most useful MC-dropout proxy for epistemic/model uncertainty.

Do **not** compare the raw magnitudes of these scores. Their scales have different meanings. Compare:

1. error-detection ROC AUC and average precision;
2. correct-versus-incorrect separation;
3. error rate across uncertainty quintiles;
4. risk-coverage behaviour;
5. association with number of available modalities;
6. correlation between the uncertainty rankings.

A useful result is not only "MC dropout is higher." It may be that MC mutual information and conflict-aware uncertainty identify **different** difficult participants. Low correlation with both retaining useful error-detection performance would suggest that the two methods capture complementary sources of uncertainty.

### Attribution

The optional leave-one-branch-out section is a **sensitivity attribution**:

- `ABS_DELTA_P_PMCI` asks how much the prediction changes when an observed branch is removed.
- positive `DELTA_MUTUAL_INFORMATION` asks whether removing that branch makes the model's MC-dropout epistemic proxy larger;
- negative `DELTA_MUTUAL_INFORMATION` means uncertainty fell after removal, which can happen if that branch was noisy or in tension with the other evidence.

These quantities are not causal estimates of a modality's clinical importance.

### Recommended run sequence

First run the notebook unchanged in pilot mode (`fold 0`, 10 MC samples). Then set `PILOT_MODE=False`, rerun from the top, and use 30 MC samples over all five folds. If the conclusions are close and a more stable estimate may require a more stable final estimate, increase to 50 samples. Only after the main comparison is working should you enable LOBO attribution.


## 12. One additional UQ baseline worth adding later: seed ensemble uncertainty

You already have independently trained seed models (17, 42, 73) for the newer seed-sensitivity experiments. That gives you another strong uncertainty baseline **without retraining**:

- align the three seed predictions for the same held-out participant;
- average the pMCI probabilities;
- calculate between-seed standard deviation, predictive entropy, expected entropy, and mutual information;
- evaluate those scores with exactly the same error-detection and risk-coverage code above.

This would give the comparison a useful structure:

**evidential strength → conflict-aware evidence → MC dropout → independently trained model ensemble.**

Do not pool three predictions for the same participant as if they were three independent participants. Keep the participant as the unit of analysis.


## 13. Reproducibility note

The analysis creates a new comparison directory and does not overwrite the existing dissertation experiment outputs. Keep the generated CSV files with the supporting material, record the checkpoint/input version used, and document any AI-assisted code generation according to the programme's GenAI declaration requirements.
